In [2]:
import pandas as pd 

In [3]:
df=pd.read_parquet('C:\\Users\\hp\\OneDrive\\Bureau\\personal_projects\\flight-delay-predictor\\data\\raw\\Combined_Flights_2019.parquet')


In [8]:
pd.set_option('display.max_columns', None)
df.head(5)

,FlightDate,Airline,Origin,Dest,CRSDepTime,Distance,Month,DayOfWeek,Cancelled,Diverted,DepDel15
2706,2019-04-01,Envoy Air,ORD,ATL,1635,606.0,4,1,False,False,0.0
2707,2019-04-02,Envoy Air,ORD,ATL,1635,606.0,4,2,False,False,1.0
2708,2019-04-03,Envoy Air,ORD,ATL,1635,606.0,4,3,False,False,0.0
2709,2019-04-04,Envoy Air,ORD,ATL,1635,606.0,4,4,False,False,1.0
2710,2019-04-05,Envoy Air,ORD,ATL,1635,606.0,4,5,False,False,1.0


In [4]:
df.shape


(8091684, 61)

In [ ]:
#only needed columns
df = df[["FlightDate", "Airline", "Origin", "Dest", "CRSDepTime",
         "Distance", "Month", "DayOfWeek", "Cancelled", "Diverted", "DepDel15"]]
df.head(5)

(8091684, 11)

In [6]:
#filtering airports based on high volume flights , diversity in geographie/climates , 5 airports as a managable number 
airports = ["ORD", "ATL", "JFK", "DFW", "LAX"]
df = df[df["Origin"].isin(airports) & df["Dest"].isin(airports)]
df.shape

(131738, 11)

In [ ]:
# out of scope 
df = df[(df["Cancelled"] == False) & (df["Diverted"] == False)]
df = df.drop(columns=["Cancelled", "Diverted"])
df.shape

In [14]:
print(df.columns.tolist())

['FlightDate', 'Airline', 'Origin', 'Dest', 'CRSDepTime', 'Distance', 'Month', 'DayOfWeek', 'DepDel15']


In [15]:
df.shape

(129453, 9)

In [ ]:
df_raw = pd.read_parquet("C:\\Users\\hp\\OneDrive\\Bureau\\personal_projects\\flight-delay-predictor\\data\\raw\\Combined_Flights_2019.parquet", columns=["Cancelled", "Diverted"])
print(df_raw["Cancelled"].value_counts())
print(df_raw["Diverted"].value_counts())
#2% cancelled , 0.3% diverted , so we can drop them from the dataset as they are out of scope for our project

Cancelled
False    7938055
True      153629
Name: count, dtype: int64
Diverted
False    8070893
True       20791
Name: count, dtype: int64


In [ ]:
KEEP_COLUMNS = [
    "FlightDate", "Airline", "Origin", "Dest", "CRSDepTime",
    "Distance", "Month", "DayOfWeek", "Cancelled", "Diverted", "DepDel15"
]

df = pd.read_parquet(
    r"C:\Users\hp\OneDrive\Bureau\personal_projects\flight-delay-predictor\data\raw\Combined_Flights_2019.parquet",
    columns=KEEP_COLUMNS
)


In [25]:
df.shape


(8091684, 11)

In [26]:
df = df[(~df["Cancelled"]) & (~df["Diverted"])]
df = df.drop(columns=["Cancelled", "Diverted"])
df = df.dropna(subset=["DepDel15"])

print(df.shape)

(7917264, 9)


In [27]:
airports = ["ORD", "ATL", "JFK", "DFW", "LAX"]
df = df[df["Origin"].isin(airports) & df["Dest"].isin(airports)]
print(df.shape)

(129453, 9)


In [28]:
df["DepDel15"].value_counts(normalize=True)

DepDel15
0.0    0.792195
1.0    0.207805
Name: proportion, dtype: float64

In [29]:
#feauture engineering
df["dep_hour"] = (df["CRSDepTime"] // 100).astype(int)
#time normalization 
route_avg = df.groupby(["Origin", "Dest"])["DepDel15"].mean().rename("route_avg_delay")
df = df.merge(route_avg, on=["Origin", "Dest"], how="left")

features = df[["Origin", "Dest", "Airline", "dep_hour", "Month",
                "DayOfWeek", "Distance", "route_avg_delay", "DepDel15"]]

print(features.shape)
features.head()

(129453, 9)


,Origin,Dest,Airline,dep_hour,Month,DayOfWeek,Distance,route_avg_delay,DepDel15
0,ORD,ATL,Envoy Air,16,4,1,606.0,0.20213,0.0
1,ORD,ATL,Envoy Air,16,4,2,606.0,0.20213,1.0
2,ORD,ATL,Envoy Air,16,4,3,606.0,0.20213,0.0
3,ORD,ATL,Envoy Air,16,4,4,606.0,0.20213,1.0
4,ORD,ATL,Envoy Air,16,4,5,606.0,0.20213,1.0
